[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc6_llms/exercices/seance1_exercices.ipynb)

# Séance 6.1 — Science des données et LLMs

**Exercices** · durée : 4h (2h de cours, 2h d'atelier)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- expliquer simplement le fonctionnement et les limites d'un LLM
- distinguer LLM, RAG, tool calling, agent et MCP
- appeler un LLM depuis Python et l'utiliser sur des données textuelles
- vérifier ses prédictions plutôt que les croire sur parole
- donner des fonctions Python au modèle comme outils, et comprendre comment un agent les enchaîne

In [ ]:
%pip install -q "transformers>=4.45,<5" "google-genai==2.9.0"

Si Colab demande de redémarrer la session après l'installation, redémarrez-la puis reprenez à partir de la cellule suivante

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc6_llms/data/"

In [ ]:
df = pd.read_csv(BASE + "avis.csv")

df.head()

# 2. Utiliser un modèle NLP spécialisé pré-entraîné

Nous allons commencer sans utiliser d'IA générative

Le modèle choisi est

`nlptown/bert-base-multilingual-uncased-sentiment`

Il s'agit d'un modèle **BERT multilingue** spécialisé dans l'analyse de sentiment d'avis produits

Le modèle a été fine-tuné sur des avis dans six langues, dont le français

Pour un texte, il prédit une note entre **1 et 5 étoiles**

## 2.1. Pré-entraînement, fine-tuning et inférence

Nous retrouvons trois notions liées à la partie A du cours

### Pré-entraînement

BERT apprend d'abord des représentations générales du langage sur de grandes quantités de texte

### Fine-tuning

Le modèle est ensuite entraîné sur une tâche plus précise, ici prédire le sentiment d'avis produits

### Inférence

Dans ce cours, nous ne réentraînons pas le modèle

Nous lui donnons simplement de nouveaux avis afin d'obtenir ses prédictions

## 2.2. Charger le modèle avec Hugging Face

La fonction `pipeline()` permet d'utiliser directement un modèle déjà entraîné

Le modèle est téléchargé lors de la première exécution puis utilisé dans l'environnement Colab

Aucune clé API n'est nécessaire pour cette partie du cours

In [ ]:
from transformers import pipeline

bert_model = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=-1
)

## 2.3. Faire une première prédiction

Testons le modèle sur un avis simple

In [ ]:
text = "Très bon produit, je suis vraiment satisfait"

result = bert_model(text)

print(result)

Le modèle renvoie notamment

- une note prédite entre 1 et 5 étoiles
- un score associé à cette prédiction

Nous allons maintenant l'utiliser sur toute notre base

# 3. Classifier tous les avis avec BERT

La pipeline Hugging Face accepte directement une liste de textes

Nous pouvons donc classifier les avis par lots

In [ ]:
bert_results = bert_model(
    df["text"].tolist(),
    batch_size=16,
    truncation=True
)

bert_results[:5]

Nous extrayons ensuite la note prédite dans chaque résultat

In [ ]:
bert_stars = []

for result in bert_results:

    stars = int(result["label"].split()[0])

    bert_stars.append(stars)

df["bert_stars"] = bert_stars

## 3.1. Transformer les étoiles en sentiment

Pour faciliter la comparaison avec notre vérité terrain, nous utilisons la règle suivante

- 1 ou 2 étoiles correspond à un sentiment négatif
- 3 étoiles correspond à un sentiment neutre
- 4 ou 5 étoiles correspond à un sentiment positif

Un avis prédit neutre sera donc considéré comme incorrect lorsque le vrai sentiment est positif ou négatif

In [ ]:
def stars_to_sentiment(stars):

    if stars <= 2:
        return "negatif"

    if stars >= 4:
        return "positif"

    return "neutre"


df["bert_sentiment"] = df["bert_stars"].apply(
    stars_to_sentiment
)

df[
    ["text", "rating", "type", "sentiment", "bert_stars", "bert_sentiment"]
].head(10)

### Exercice 1 : évaluer BERT

Nous connaissons le vrai sentiment grâce à la colonne `sentiment`

Créez une colonne `bert_ok` qui vaut `True` lorsque la prédiction de BERT est correcte puis calculez l'accuracy globale

## 3.2. Regarder la performance selon le type d'avis

Une accuracy globale peut masquer des différences importantes entre les groupes

### Questions

- BERT obtient-il la même performance sur les trois types d'avis?
- Quel groupe semble le plus difficile?
- Certains avis sarcastiques sont-ils correctement classés malgré tout?

In [ ]:
df[
    df["type"] == "sarcastique"
][
    ["text", "sentiment", "bert_stars", "bert_sentiment"]
].head(10)

BERT prend en compte le contexte du texte.

La question intéressante n'est donc pas de savoir si « NLP classique » comprend ou non le sarcasme

Nous allons plutôt comparer un **modèle spécialisé de classification** à un **LLM génératif généraliste**

# 4. Utiliser Gemini comme classifieur

Nous allons maintenant demander à Gemini de réaliser la même tâche

Contrairement à BERT dans la partie précédente, Gemini sera appelé via une API

Nous utiliserons `gemini-3.5-flash-lite`, un modèle adapté aux traitements courts et aux workflows à volume relativement élevé

In [ ]:
from google import genai
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")

ai = genai.Client(api_key=key)

gemini_model = "gemini-3.5-flash-lite"

## 4.1. Premier appel API

Le programme envoie maintenant une requête à un modèle exécuté à distance

In [ ]:
question = (
    "Classe le sentiment réel de cet avis client "
    "en tenant compte du contexte et d'un éventuel sarcasme "
    "Réponds uniquement par positif ou negatif\n\n"
    "Génial, il est tombé en panne après seulement deux jours"
)

result = ai.interactions.create(
    model=gemini_model,
    input=question
)

print(result.output_text)

Le fonctionnement est différent de BERT

| BERT dans ce cours | Gemini dans ce cours |
|---|---|
| Modèle spécialisé en classification | LLM génératif généraliste |
| Modèle téléchargé dans Colab | Modèle exécuté à distance |
| Pas d'appel API pour chaque prédiction | Appel API vers Gemini |
| Sortie prévue à l'avance en étoiles | Tâche formulée avec une instruction |

## 4.2. Créer une fonction de classification avec Gemini

La fonction suivante reçoit un texte puis demande à Gemini de répondre uniquement par `positif` ou `negatif`

Elle effectue plusieurs tentatives si le service est temporairement indisponible

In [ ]:
import time

def sentiment_gemini(text):

    question = (
        "Classe le sentiment réel de cet avis client "
        "en tenant compte du contexte et d'un éventuel sarcasme "
        "Réponds uniquement par positif ou negatif\n\n"
        + text
    )

    for attempt in range(3):

        try:

            result = ai.interactions.create(
                model=gemini_model,
                input=question
            )

            answer = result.output_text.strip().lower()

            if "positif" in answer:
                return "positif"

            if "negatif" in answer or "négatif" in answer:
                return "negatif"

            return "invalide"

        except Exception as error:

            if attempt == 2:
                raise error

            print("Service temporairement indisponible")
            print("Nouvelle tentative dans 5 secondes")

            time.sleep(5)

# 5. Comparer BERT et Gemini

Pour limiter le nombre d'appels API pendant le cours, nous utilisons un échantillon équilibré de 15 avis

- 5 avis positifs
- 5 avis négatifs
- 5 avis sarcastiques

BERT a été évalué sur toute la base

La comparaison directe entre BERT et Gemini sera réalisée uniquement sur cet échantillon commun

In [ ]:
test = (
    df.groupby("type", group_keys=False)
      .sample(n=5, random_state=10)
      .reset_index(drop=True)
      .copy()
)

test[
    ["text", "type", "sentiment", "bert_sentiment"]
]

## Exercice 2 : classifier l'échantillon avec Gemini

## 5.1. Comparer les performances

Nous créons un indicateur de bonne prédiction pour chaque modèle

## 5.2. Comparer selon le type d'avis

### Questions

- Les deux modèles ont-ils la même performance globale?
- La différence est-elle la même pour tous les types d'avis?
- Que se passe-t-il sur les avis sarcastiques?
- BERT réussit-il certains cas où Gemini échoue?
- Gemini réussit-il certains cas où BERT échoue?

# 6. Étudier les désaccords entre BERT et Gemini

L'accuracy ne nous dit pas tout

Les observations sur lesquelles les deux modèles ne sont pas d'accord sont particulièrement intéressantes

## Exercice 3 : analyser un désaccord

Choisissez un avis sarcastique mal classé par au moins un des deux modèles

Demandez ensuite à Gemini d'expliquer en deux phrases les éléments du texte qui permettent d'identifier le sentiment réel

### Ce que montre cette comparaison

BERT et Gemini sont tous les deux fondés sur des Transformers, mais ils n'ont pas le même rôle

BERT est ici spécialisé pour une tâche précise de classification

Gemini est un modèle génératif généraliste auquel nous décrivons la tâche avec une instruction

Le sarcasme constitue un cas intéressant car le sentiment réel peut dépendre fortement de la relation entre plusieurs éléments de la phrase

Aucun des deux modèles n'est garanti de réussir tous les exemples

# 7. Donner une fonction Python au LLM avec le tool calling

Nous allons maintenant changer de rôle

Gemini ne doit plus seulement classifier un texte

Nous voulons qu'il puisse demander à Python de calculer une statistique

Question

> Quelle est l'accuracy de BERT sur les avis sarcastiques ?

Le chiffre doit être calculé à partir du DataFrame et non inventé par le LLM

In [ ]:
def bert_accuracy(kind):

    data = df[df["type"] == kind]

    value = (
        data["bert_sentiment"]
        == data["sentiment"]
    ).mean()

    return round(float(value), 2)

## 7.1. Décrire l'outil au modèle

Nous indiquons au modèle

- le nom de la fonction
- son objectif
- le paramètre qu'elle accepte

Cette description permet au modèle de déterminer si l'outil est utile

In [ ]:
bert_accuracy_tool = {
    "type": "function",
    "name": "bert_accuracy",
    "description": (
        "Calcule l'accuracy de BERT "
        "pour un type d'avis"
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "kind": {
                "type": "string",
                "enum": [
                    "positif",
                    "negatif",
                    "sarcastique"
                ]
            }
        },
        "required": ["kind"]
    }
}

## 7.2. Laisser Gemini choisir l'outil

À cette étape, Gemini ne calcule pas lui-même l'accuracy

Il détermine seulement quelle fonction appeler et avec quel argument

In [ ]:
interaction = ai.interactions.create(
    model=gemini_model,
    input=(
        "Quelle est l'accuracy de BERT "
        "sur les avis sarcastiques ?"
    ),
    tools=[bert_accuracy_tool]
)

call = next(
    step
    for step in interaction.steps
    if step.type == "function_call"
)

print("Fonction demandée :", call.name)
print("Arguments :", call.arguments)

## 7.3. Python exécute la fonction

In [ ]:
value = bert_accuracy(**call.arguments)

print("Résultat calculé par Python :", value)

## 7.4. Renvoyer le résultat au modèle

In [ ]:
import json

final = ai.interactions.create(
    model=gemini_model,
    previous_interaction_id=interaction.id,
    tools=[bert_accuracy_tool],
    input=[
        {
            "type": "function_result",
            "name": call.name,
            "call_id": call.id,
            "result": [
                {
                    "type": "text",
                    "text": json.dumps(value)
                }
            ]
        }
    ]
)

print(final.output_text)

### À retenir

Le tool calling suit ici quatre étapes

1. Gemini comprend la question et choisit un outil
2. Python exécute réellement la fonction
3. le résultat est renvoyé à Gemini
4. Gemini formule la réponse

Nous n'avons pas encore de boucle de décision multi-étapes

Il s'agit donc de **tool calling**

# 8. Préparer plusieurs outils pour un agent

Nous allons maintenant donner à Gemini plusieurs capacités d'analyse

- comparer les performances globales de BERT et Gemini
- comparer leurs performances selon le type d'avis
- consulter les observations sur lesquelles les deux modèles ne sont pas d'accord

In [ ]:
def global_accuracy():

    return {
        "bert": round(float(test["bert_ok"].mean()), 2),
        "gemini": round(float(test["gemini_ok"].mean()), 2)
    }


def accuracy_by_type():

    table = (
        test.groupby("type")[
            ["bert_ok", "gemini_ok"]
        ]
        .mean()
        .round(2)
    )

    result = {}

    for kind in table.index:

        result[kind] = {
            "bert": float(table.loc[kind, "bert_ok"]),
            "gemini": float(table.loc[kind, "gemini_ok"])
        }

    return result


def show_disagreements(kind):

    data = test[
        (test["type"] == kind)
        & (
            test["bert_sentiment"]
            != test["gemini_sentiment"]
        )
    ]

    return data[
        [
            "text",
            "sentiment",
            "bert_sentiment",
            "gemini_sentiment"
        ]
    ].to_dict("records")

## 8.1. Décrire les outils

Cette partie technique est fournie dans le cours

L'objectif est de comprendre ce que ces descriptions permettent au modèle de savoir

In [ ]:
tool_defs = [
    {
        "type": "function",
        "name": "global_accuracy",
        "description": (
            "Compare les accuracies globales "
            "de BERT et Gemini"
        ),
        "parameters": {
            "type": "object",
            "properties": {}
        }
    },
    {
        "type": "function",
        "name": "accuracy_by_type",
        "description": (
            "Compare les accuracies de BERT et Gemini "
            "pour les avis positifs, négatifs et sarcastiques"
        ),
        "parameters": {
            "type": "object",
            "properties": {}
        }
    },
    {
        "type": "function",
        "name": "show_disagreements",
        "description": (
            "Retourne les avis d'un type donné "
            "sur lesquels BERT et Gemini ne sont pas d'accord"
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "kind": {
                    "type": "string",
                    "enum": [
                        "positif",
                        "negatif",
                        "sarcastique"
                    ]
                }
            },
            "required": ["kind"]
        }
    }
]

functions = {
    "global_accuracy": global_accuracy,
    "accuracy_by_type": accuracy_by_type,
    "show_disagreements": show_disagreements
}

# 9. Construire un véritable agent multi-étapes

Cette fois, nous voulons permettre au modèle de

1. choisir une première analyse
2. observer le résultat
3. décider si une autre analyse est nécessaire
4. choisir éventuellement un nouvel outil
5. recommencer jusqu'à obtenir assez d'informations

C'est cette boucle qui distingue l'agent du simple tool calling

In [ ]:
agent_rules = (
    "Tu es un assistant de Data Science "
    "Tu compares BERT et Gemini pour classifier des avis clients "
    "N'invente jamais de chiffre "
    "Utilise les outils pour obtenir les résultats "
    "Utilise au maximum un outil par étape "
    "Après chaque résultat, décide si une autre analyse est nécessaire "
    "Si un type d'avis semble expliquer une différence de performance, "
    "consulte les désaccords correspondants "
    "Réponds en français et reste concis"
)

## 9.1. La boucle agentique

Le code suivant est fourni

Il réalise successivement

1. un appel à Gemini
2. la détection d'une éventuelle demande d'outil
3. l'exécution de la fonction Python
4. le renvoi du résultat à Gemini
5. une nouvelle décision du modèle

`max_steps` limite le nombre d'étapes afin d'éviter une boucle sans fin

In [ ]:
def run_agent(question, max_steps=5):

    interaction = ai.interactions.create(
        model=gemini_model,
        input=question,
        tools=tool_defs,
        system_instruction=agent_rules
    )

    for step_number in range(max_steps):

        calls = [
            step
            for step in interaction.steps
            if step.type == "function_call"
        ]

        if len(calls) == 0:
            return interaction.output_text

        call = calls[0]

        print(
            "Étape",
            step_number + 1,
            "- outil :",
            call.name
        )

        value = functions[call.name](
            **call.arguments
        )

        print("Résultat :", value)
        print()

        interaction = ai.interactions.create(
            model=gemini_model,
            previous_interaction_id=interaction.id,
            tools=tool_defs,
            system_instruction=agent_rules,
            input=[
                {
                    "type": "function_result",
                    "name": call.name,
                    "call_id": call.id,
                    "result": [
                        {
                            "type": "text",
                            "text": json.dumps(
                                value,
                                ensure_ascii=False,
                                default=str
                            )
                        }
                    ]
                }
            ]
        )

    return "Arrêt après le nombre maximal d'étapes"

### Pourquoi s'agit-il maintenant d'un agent ?

La séquence des outils n'est pas fixée à l'avance

Le modèle reçoit le résultat d'une action puis choisit la suivante

La logique devient donc

**décision → action → observation → nouvelle décision**

## Exercice 4 : laisser l'agent analyser les résultats

Demandez à l'agent de répondre à la question suivante

> Compare BERT et Gemini sur notre échantillon  
> Détermine si l'écart de performance vient surtout d'un type d'avis  
> Si un type semble particulièrement intéressant, consulte les désaccords entre les deux modèles  
> Termine par une conclusion courte sur l'intérêt potentiel du LLM pour cette tâche

### Questions

- quel outil l'agent utilise-t-il en premier?
- quelle information obtient-il?
- cette information influence-t-elle l'action suivante?
- consulte-t-il les résultats par type d'avis?
- consulte-t-il des exemples de désaccord?
- sa conclusion est-elle cohérente avec les données?

## Exercice 5 : ajouter un nouvel outil à l'agent

Nous voulons maintenant permettre à l'agent de comparer les notes attribuées par les clients

Créez une fonction `rating_by_type()` qui retourne la note moyenne pour chaque type d'avis

Ajoutez ensuite cette fonction à la liste des fonctions disponibles et à la description des outils

# 10. Que nous apporte chaque approche ?

| Approche | Rôle dans le cours | Point fort | Limite |
|---|---|---|---|
| **BERT spécialisé** | Classifier le sentiment | Modèle conçu pour une tâche précise et exécution locale | Peut échouer sur des formulations inhabituelles ou ambiguës |
| **Gemini** | Comprendre et classifier en suivant une instruction | Modèle généraliste capable de traiter une grande variété de formulations | Peut se tromper et nécessite ici des appels API |
| **Agent** | Orchestrer plusieurs analyses | Choisit des outils et adapte les actions aux résultats | Plus complexe et peut multiplier les appels |
| **Python direct** | Réaliser les calculs déterministes | Simple, reproductible et précis | Ne choisit pas seul quelle analyse réaliser |

# 11. Limites de la comparaison

Notre exercice doit être interprété avec prudence:

- la base est fictive
- elle contient peu d'avis
- seuls 15 avis sont directement comparés entre BERT et Gemini
- le modèle BERT utilisé a été spécialisé sur des avis produits mais pas spécifiquement sur le sarcasme
- le prompt envoyé à Gemini influence ses réponses
- les réponses d'un LLM peuvent varier
- un autre modèle spécialisé pourrait produire des résultats différents

Le cours montre donc un **workflow de comparaison et d'évaluation**, pas une preuve générale de supériorité d'un type de modèle

# Conclusion

Dans ce cours, nous sommes progressivement passés d'un LLM isolé à des systèmes capables d'utiliser des informations, des outils et des boucles de décision

### Comprendre

Un LLM génère du langage à partir du contexte qu'il reçoit

Ses réponses peuvent être utiles mais doivent être vérifiées

### Augmenter

| Besoin | Mécanisme |
|---|---|
| Apporter une information externe | **RAG** |
| Utiliser une fonction ou un service | **Tools et function calling** |
| Choisir et enchaîner plusieurs actions | **Agent** |
| Standardiser l'accès aux outils et ressources | **MCP** |

### Appliquer en Data Science

La partie C a permis de comparer un modèle NLP spécialisé pré-entraîné à un LLM génératif sur une même tâche

Nous avons vu que

- un modèle spécialisé peut être utilisé directement sans nouvel entraînement
- un LLM peut réaliser une tâche de classification à partir d'une instruction
- les performances doivent être mesurées sur des données labellisées
- les cas de désaccord sont souvent plus informatifs que l'accuracy seule
- Python reste préférable pour les calculs déterministes
- le tool calling permet au modèle de demander un calcul fiable
- un agent ajoute une boucle permettant de choisir plusieurs actions successives

### Idée finale

> Le bon usage de l'IA générative ne consiste pas à remplacer systématiquement les méthodes existantes mais à identifier les tâches pour lesquelles elle apporte une capacité supplémentaire et à vérifier cette valeur avec des données

# Sources et ressources

## Ressource pédagogique principale

**Stéphane Robert — Panorama IA : LLM, RAG, agents et MCP**  
https://blog.stephane-robert.info/docs/developper/programmation/python/ia-panorama/

## Cours universitaires

**Stanford — CS336: Language Modeling from Scratch**  
https://cs336.stanford.edu/

**UC Berkeley — Large Language Model Agents**  
https://rdi.berkeley.edu/llm-agents/

**Harvard — CS50's Introduction to Artificial Intelligence with Python**  
https://cs50.harvard.edu/ai/

**Carnegie Mellon — Large Language Model Applications**  
https://cmu-llms.org/

## Références académiques

**Vaswani et al. — Attention Is All You Need**  
https://arxiv.org/abs/1706.03762

**Devlin et al. — BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding**  
https://arxiv.org/abs/1810.04805

**Lewis et al. — Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks**  
https://arxiv.org/abs/2005.11401

**Yao et al. — ReAct: Synergizing Reasoning and Acting in Language Models**  
https://arxiv.org/abs/2210.03629

**Schick et al. — Toolformer: Language Models Can Teach Themselves to Use Tools**  
https://arxiv.org/abs/2302.04761

## Modèle NLP utilisé dans le cours

**NLP Town — bert-base-multilingual-uncased-sentiment**  
https://huggingface.co/nlptown/bert-base-multilingual-uncased-sentiment

**Hugging Face Transformers — Pipelines**  
https://huggingface.co/docs/transformers/main_classes/pipelines

## Gemini API

**Interactions API**  
https://ai.google.dev/gemini-api/docs/interactions-overview

**Function calling**  
https://ai.google.dev/gemini-api/docs/function-calling

**Gemini 3.5 Flash-Lite**  
https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash-lite

**Pricing**  
https://ai.google.dev/gemini-api/docs/pricing

## MCP

**Anthropic — Introducing the Model Context Protocol**  
https://www.anthropic.com/news/model-context-protocol

**Documentation officielle MCP**  
https://modelcontextprotocol.io/

## Agents de programmation

**OpenAI — Codex CLI**  
https://help.openai.com/en/articles/11096431

**Anthropic — Claude Code**  
https://code.claude.com/docs/en/getting-started

**Google — Gemini CLI**  
https://github.com/google-gemini/gemini-cli

## Google Colab

**Google — Colab FAQ**  
https://research.google.com/colaboratory/faq.html

## Impact environnemental

**International Energy Agency — Energy and AI**  
https://www.iea.org/reports/energy-and-ai

**International Energy Agency — Key Questions on Energy and AI**  
https://www.iea.org/reports/key-questions-on-energy-and-ai

## Données utilisées dans le cours

Le fichier `avis.csv` a été créé spécifiquement pour ce cours

Les avis sont fictifs et ne constituent pas un benchmark scientifique